# Thai Math VQA LLaVA/Ollama Colab Notebook

This notebook downloads the Kaggle Thai Math VQA competition data into Google Colab, runs local Ollama/LLaVA inference, and writes `/content/math-vqa-output/submission.csv` plus `/content/math-vqa-output/raw_predictions.csv`.

Hugging Face fallback: if Ollama cannot be installed or cannot pull `llava`, attach a Kaggle dataset containing a compatible vision-language model such as `llava-hf/llava-1.5-7b-hf` or `Qwen/Qwen2.5-VL-3B-Instruct`, then replace `query_llava(...)` in the inference cells with the corresponding Transformers pipeline call. Keep `clean_model_answer(...)`, `PredictionRecord`, and `write_outputs(...)` unchanged.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

import requests

COMPETITION_SLUG = "super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen"
IS_COLAB = Path("/content").exists()
DATA_PARENT = Path("/content/math-vqa-data") if IS_COLAB else Path("../data").resolve()
DATA_ROOT = DATA_PARENT / COMPETITION_SLUG
OUTPUT_DIR = Path("/content/math-vqa-output") if IS_COLAB else Path("../outputs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = os.getenv("OLLAMA_MODEL", "llava")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_REQUEST_TIMEOUT = int(os.getenv("OLLAMA_REQUEST_TIMEOUT", "600"))
OLLAMA_RETRIES = int(os.getenv("OLLAMA_RETRIES", "1"))

print(f"IS_COLAB={IS_COLAB}")
print(f"DATA_ROOT={DATA_ROOT}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"OLLAMA_REQUEST_TIMEOUT={OLLAMA_REQUEST_TIMEOUT}")
print(f"OLLAMA_RETRIES={OLLAMA_RETRIES}")

def run_command(command, *, shell=False, timeout=None):
    return subprocess.run(
        command,
        shell=shell,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )

def run_checked(command, *, shell=False, timeout=None):
    result = run_command(command, shell=shell, timeout=timeout)
    if result.returncode != 0:
        rendered_command = command if isinstance(command, str) else " ".join(command)
        raise RuntimeError(
            f"Command failed: {rendered_command}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )
    return result

def ensure_zstd_for_ollama():
    if shutil.which("zstd") is not None:
        print("zstd is available.")
        return
    if shutil.which("apt-get") is None:
        raise RuntimeError(
            "Ollama install requires zstd, but `zstd` is not on PATH and `apt-get` is unavailable. "
            "Use a Kaggle/runtime image that includes zstd, package Ollama/model assets as a dataset, "
            "or switch this notebook to the documented Hugging Face fallback."
        )
    print("Installing zstd because the Ollama installer requires it for .tar.zst extraction.")
    run_checked(
        "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends zstd",
        shell=True,
        timeout=600,
    )
    if shutil.which("zstd") is None:
        raise RuntimeError(
            "Installed the zstd apt package, but the `zstd` command is still not on PATH. "
            "Ollama cannot be installed in this runtime; use packaged assets or the Hugging Face fallback."
        )

def data_files_ready(root):
    required_paths = [
        root / "train.csv",
        root / "test.csv",
        root / "sample_submission.csv",
        root / "images" / "images",
    ]
    return all(path.exists() for path in required_paths)

def configure_kaggle_credentials():
    kaggle_dir = Path.home() / ".kaggle"
    token_path = kaggle_dir / "kaggle.json"
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    username = os.getenv("KAGGLE_USERNAME")
    key = os.getenv("KAGGLE_KEY")
    if username and key:
        token_path.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
        token_path.chmod(0o600)
        print("Configured Kaggle credentials from KAGGLE_USERNAME/KAGGLE_KEY.")
        return

    if token_path.exists():
        token_path.chmod(0o600)
        print(f"Using existing Kaggle credential file at {token_path}.")
        return

    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError(
            "Kaggle credentials are required to download competition data. "
            "Set KAGGLE_USERNAME and KAGGLE_KEY, or place kaggle.json at ~/.kaggle/kaggle.json."
        ) from exc

    print("Upload your Kaggle API token file named kaggle.json.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("Expected an uploaded file named kaggle.json.")
    token_path.write_bytes(uploaded["kaggle.json"])
    token_path.chmod(0o600)
    print(f"Saved Kaggle credentials to {token_path}.")

def extract_competition_archive(archive_path):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_ROOT)
    if not data_files_ready(DATA_ROOT):
        raise RuntimeError(
            f"Downloaded archive extracted to {DATA_ROOT}, but required train/test/sample/image files were not found."
        )

def download_competition_data():
    if data_files_ready(DATA_ROOT):
        print(f"Competition data already exists at {DATA_ROOT}.")
        return

    configure_kaggle_credentials()
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    command_variants = [
        ["kaggle", "competitions", "download", "-c", COMPETITION_SLUG, "-p", str(DATA_PARENT), "-o"],
        ["kaggle", "competitions", "download", COMPETITION_SLUG, "-p", str(DATA_PARENT), "-o"],
    ]
    failures = []
    for command in command_variants:
        result = run_command(command, timeout=1800)
        if result.returncode == 0:
            break
        failures.append(" ".join(command) + "\nSTDOUT:\n" + result.stdout + "\nSTDERR:\n" + result.stderr)
    else:
        raise RuntimeError(
            "Kaggle competition data download failed. Confirm that your Kaggle API token is valid "
            "and that you accepted the competition rules on Kaggle.\n\n" + "\n\n".join(failures)
        )

    archive_candidates = [
        DATA_PARENT / f"{COMPETITION_SLUG}.zip",
        DATA_PARENT / "super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen.zip",
    ]
    archive_candidates.extend(sorted(DATA_PARENT.glob("*.zip")))
    archive_path = next((candidate for candidate in archive_candidates if candidate.exists()), None)
    if archive_path is None:
        raise RuntimeError(f"Kaggle download completed, but no zip archive was found in {DATA_PARENT}.")
    extract_competition_archive(archive_path)
    print(f"Downloaded and extracted competition data to {DATA_ROOT}.")

run_checked([sys.executable, "-m", "pip", "install", "-q", "kaggle", "pillow", "requests", "tqdm", "pandas"])
download_competition_data()

if shutil.which("ollama") is None:
    ensure_zstd_for_ollama()
    run_checked(
        "bash -lc 'set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'",
        shell=True,
        timeout=600,
    )
if shutil.which("ollama") is None:
    raise RuntimeError("Ollama install failed: the `ollama` command is still not on PATH after install.")

server_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

ready = False
for _ in range(60):
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=2)
        if response.ok:
            ready = True
            break
    except requests.RequestException:
        time.sleep(2)
if not ready:
    raise RuntimeError("Ollama setup failed: server did not become ready on http://localhost:11434")

# Equivalent shell command: ollama pull $OLLAMA_MODEL
run_checked(["ollama", "pull", MODEL_NAME], timeout=1800)
print("Ollama is ready.")


In [ ]:
from pathlib import Path
import sys

RUNTIME_SRC = Path("/content/math_vqa_runtime") if Path("/content").exists() else Path.cwd() / "math_vqa_runtime"
MATERIALIZED_MODULES = {'math_vqa/__init__.py': '"""Thai Math VQA Kaggle inference helpers."""\n\n__version__ = "0.1.0"\n', 'math_vqa/data.py': 'from __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport re\n\nimport pandas as pd\n\n\nCOMPETITION_DIR_NAME = "super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen"\nTHAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")\n\n\n@dataclass(frozen=True)\nclass CompetitionPaths:\n    root: Path\n\n    @property\n    def train_csv(self) -> Path:\n        return self.root / "train.csv"\n\n    @property\n    def test_csv(self) -> Path:\n        return self.root / "test.csv"\n\n    @property\n    def sample_submission_csv(self) -> Path:\n        return self.root / "sample_submission.csv"\n\n    @classmethod\n    def auto(cls) -> "CompetitionPaths":\n        kaggle_root = Path("/kaggle/input/competitions") / COMPETITION_DIR_NAME\n        if kaggle_root.exists():\n            return cls(kaggle_root)\n        local_root = Path(__file__).resolve().parents[2] / "data" / COMPETITION_DIR_NAME\n        return cls(local_root)\n\n\ndef load_competition_frames(paths: CompetitionPaths) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    train_df = pd.read_csv(\n        paths.train_csv,\n        dtype={"id": str, "image_path": str, "answer": str},\n        keep_default_na=False,\n    )\n    test_df = pd.read_csv(\n        paths.test_csv,\n        dtype={"id": str, "image_path": str},\n        keep_default_na=False,\n    )\n    sample_df = pd.read_csv(\n        paths.sample_submission_csv,\n        dtype={"id": str, "answer": str},\n        keep_default_na=False,\n    )\n    return train_df, test_df, sample_df\n\n\ndef resolve_image_path(paths: CompetitionPaths, image_path: str | Path) -> Path:\n    image_path = Path(image_path)\n    if image_path.is_absolute() and image_path.exists():\n        return image_path\n    candidates = [\n        paths.root / image_path,\n        paths.root / "images" / image_path,\n        paths.root / "images" / "images" / image_path.name,\n    ]\n    for candidate in candidates:\n        if candidate.exists():\n            return candidate\n    return candidates[0]\n\n\ndef validate_data_files(\n    paths: CompetitionPaths,\n    train_df: pd.DataFrame,\n    test_df: pd.DataFrame,\n    sample_df: pd.DataFrame,\n) -> None:\n    if not paths.root.exists():\n        raise ValueError(f"data root does not exist: {paths.root}")\n    for csv_path in (paths.train_csv, paths.test_csv, paths.sample_submission_csv):\n        if not csv_path.exists():\n            raise ValueError(f"missing csv file: {csv_path}")\n\n    expected_columns = {\n        "train.csv": (train_df, ["id", "image_path", "answer"]),\n        "test.csv": (test_df, ["id", "image_path"]),\n        "sample_submission.csv": (sample_df, ["id", "answer"]),\n    }\n    for name, (frame, columns) in expected_columns.items():\n        if list(frame.columns) != columns:\n            raise ValueError(f"{name} columns must be exactly {columns}, got {list(frame.columns)}")\n        if frame["id"].duplicated().any():\n            duplicated = frame.loc[frame["id"].duplicated(), "id"].tolist()\n            raise ValueError(f"{name} has duplicate ids: {duplicated[:5]}")\n\n    if set(train_df["id"]) & set(test_df["id"]):\n        raise ValueError("train and test ids must not overlap")\n    if set(sample_df["id"]) != set(test_df["id"]):\n        raise ValueError("sample_submission ids must match test ids")\n\n    missing_images: list[str] = []\n    for frame_name, frame in (("train.csv", train_df), ("test.csv", test_df)):\n        for row in frame.itertuples(index=False):\n            resolved = resolve_image_path(paths, row.image_path)\n            if not resolved.exists():\n                missing_images.append(f"{frame_name}:{row.id}:{row.image_path}")\n    if missing_images:\n        raise ValueError(f"missing image files: {missing_images[:10]}")\n\n\ndef answer_prior(train_df: pd.DataFrame) -> str:\n    answers = [str(answer).strip() for answer in train_df["answer"].tolist() if str(answer).strip()]\n    simple_answers = [\n        answer.translate(THAI_DIGITS)\n        for answer in answers\n        if re.fullmatch(r"[0-9๐-๙]+", answer)\n    ]\n    if simple_answers:\n        return Counter(simple_answers).most_common(1)[0][0]\n    if answers:\n        return Counter(answers).most_common(1)[0][0]\n    return "2"\n', 'math_vqa/evaluation.py': 'from __future__ import annotations\n\nimport re\nfrom typing import Sequence\n\n\nTHAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")\nUNIT_WORDS = [\n    "square centimeters",\n    "ตารางเซนติเมตร",\n    "ลูกบาศก์หน่วย",\n    "เซนติเมตร",\n    "degrees",\n    "years old",\n    "ร้อยละ",\n    "ดอลลาร์",\n    "องศา",\n    "หน่วย",\n    "จำนวน",\n    "วิธี",\n    "แบบ",\n    "ค่า",\n    "บาท",\n]\nLATEX_REPLACEMENTS = {\n    r"\\pi": "pi",\n    r"\\times": "*",\n    r"\\cdot": "*",\n    r"\\div": "/",\n    r"\\pm": "+-",\n    r"\\left": "",\n    r"\\right": "",\n    r"\\,": "",\n    r"\\;": "",\n    r"\\:": "",\n    r"\\!": "",\n}\n\n\ndef normalize_answer(value: object) -> str:\n    if value is None:\n        return ""\n    text = str(value).lower().strip().translate(THAI_DIGITS)\n    text = text.replace("$", "")\n    text = re.sub(r"\\\\frac\\s*\\{([^{}]+)\\}\\s*\\{([^{}]+)\\}", lambda m: f"{m.group(1)}/{m.group(2)}", text)\n    text = re.sub(r"\\\\sqrt\\s*\\{([^{}]+)\\}", lambda m: f"sqrt{m.group(1)}", text)\n    for macro in ("overrightarrow", "overline", "vec"):\n        text = re.sub(rf"\\\\{macro}\\s*\\{{([^{{}}]+)\\}}", r"\\1", text)\n    for source, replacement in LATEX_REPLACEMENTS.items():\n        text = text.replace(source, replacement)\n    for unit in sorted(UNIT_WORDS, key=len, reverse=True):\n        text = re.sub(re.escape(unit), "", text, flags=re.IGNORECASE)\n    text = re.sub(r"\\s+", "", text)\n    text = re.sub(r"[{}\\\\,]", "", text)\n    text = re.sub(r"\\(([-+]?\\d+)\\)", r"\\1", text)\n    text = re.sub(r"(?<![a-z0-9])([-+]?\\d+)\\.0+(?![a-z0-9])", r"\\1", text)\n    return text\n\n\ndef normalized_accuracy(predictions: Sequence[object], truths: Sequence[object]) -> float:\n    if len(predictions) != len(truths):\n        raise ValueError("predictions and truths must have the same length")\n    if len(predictions) == 0:\n        return 0.0\n    matches = sum(\n        normalize_answer(prediction) == normalize_answer(truth)\n        for prediction, truth in zip(predictions, truths, strict=True)\n    )\n    return matches / len(predictions)\n', 'math_vqa/postprocess.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nimport re\n\n\nTHAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")\nPREFIX_PATTERNS = [\n    r"answer",\n    r"final answer",\n    r"the answer is",\n    r"คำตอบคือ",\n    r"คำตอบ",\n    r"ตอบ",\n]\nFINAL_ANSWER_PATTERNS = [\n    re.compile(\n        r"(?:final answer)(?:\\s+in\\s+this\\s+case)?\\s+is\\s*(?:[:：]|\\s+-\\s+)?\\s*[\\"\'“”‘’]*(?P<answer>[^\\n;。]+)",\n        re.IGNORECASE,\n    ),\n    re.compile(\n        r"(?:final answer|the answer is|answer is|คำตอบคือ|ตอบ)\\s*(?:[:：]|\\s+-\\s+)?\\s*[\\"\'“”‘’]*(?P<answer>[^\\n;。]+)",\n        re.IGNORECASE,\n    ),\n]\nUNUSABLE_PATTERN = re.compile(\n    r"("\n    r"cannot|can\'t|unable|sorry|"\n    r"too small|blurry|not clear|not possible|"\n    r"provide a clearer|please provide|does not provide|no math problem|"\n    r"ไม่สามารถ|ขอโทษ|ไม่ชัด"\n    r")",\n    re.IGNORECASE,\n)\n\n\n@dataclass(frozen=True)\nclass CleanedAnswer:\n    answer: str\n    used_fallback: bool\n\n\ndef _candidate_lines(raw_text: str) -> list[str]:\n    lines: list[str] = []\n    for line in raw_text.splitlines():\n        stripped = line.strip()\n        if not stripped:\n            continue\n        if stripped.lower() in {"```", "```text", "```markdown", "```python"}:\n            continue\n        lines.append(stripped)\n    if not lines and raw_text.strip():\n        lines.append(raw_text.strip())\n    return lines\n\n\ndef _remove_prefix(text: str) -> str:\n    cleaned = text\n    for prefix in PREFIX_PATTERNS:\n        cleaned = re.sub(rf"^\\s*{prefix}\\s*[:：\\-]?\\s*", "", cleaned, flags=re.IGNORECASE)\n    return cleaned\n\n\ndef _clean_line(line: str) -> str:\n    cleaned = line.strip().strip("`")\n    cleaned = _remove_prefix(cleaned)\n    cleaned = cleaned.strip().strip("\\"\'“”‘’")\n    cleaned = cleaned.rstrip(".,;:。")\n    cleaned = cleaned.strip().strip("\\"\'“”‘’")\n    cleaned = cleaned.replace("$", "")\n    cleaned = cleaned.translate(THAI_DIGITS)\n    return cleaned.strip()\n\n\ndef _looks_like_short_answer(text: str) -> bool:\n    if not text:\n        return False\n    if text.lower() in {"is", "answer", "final answer", "คำตอบ", "ตอบ"}:\n        return False\n    if re.match(r"^\\d+\\.\\s+\\D", text):\n        return False\n    if UNUSABLE_PATTERN.search(text):\n        return False\n    if len(text) > 80:\n        return False\n    if len(text.split()) > 12:\n        return False\n    return True\n\n\ndef _trim_answer_span(text: str) -> str:\n    return re.split(r"(?<=[0-9A-Za-z๐-๙\\"\'”’])\\.\\s+", text.strip(), maxsplit=1)[0]\n\n\ndef _extract_declared_final_answer(raw_text: str) -> str | None:\n    for pattern in FINAL_ANSWER_PATTERNS:\n        for match in pattern.finditer(raw_text):\n            cleaned = _clean_line(_trim_answer_span(match.group("answer")))\n            if _looks_like_short_answer(cleaned):\n                return cleaned\n    return None\n\n\ndef clean_model_answer(raw_output: object, fallback_answer: str) -> CleanedAnswer:\n    raw_text = "" if raw_output is None else str(raw_output)\n    declared_final_answer = _extract_declared_final_answer(raw_text)\n    if declared_final_answer is not None:\n        return CleanedAnswer(answer=declared_final_answer, used_fallback=False)\n    for line in _candidate_lines(raw_text):\n        cleaned = _clean_line(line)\n        if _looks_like_short_answer(cleaned):\n            return CleanedAnswer(answer=cleaned, used_fallback=False)\n    return CleanedAnswer(answer=str(fallback_answer), used_fallback=True)\n', 'math_vqa/preprocessing.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom PIL import Image, ImageEnhance\n\n\nLOW_RES_IMAGE_IDS = {"94", "101", "134", "140", "162", "200"}\nCONTRAST_IMAGE_IDS = {"156"}\nHIGH_RES_IMAGE_IDS = {"451", "569"}\nRESAMPLE = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS\n\n\n@dataclass(frozen=True)\nclass PreprocessResult:\n    image: Image.Image\n    name: str\n    final_size: tuple[int, int]\n\n\ndef load_rgb_image(image_path: str | Path) -> Image.Image:\n    with Image.open(image_path) as image:\n        return image.convert("RGB")\n\n\ndef _resize_by_max_side(image: Image.Image, max_side: int, upscale: bool) -> Image.Image:\n    width, height = image.size\n    current_max = max(width, height)\n    if current_max == 0:\n        raise ValueError("image has zero-sized dimension")\n    if current_max == max_side:\n        return image.copy()\n    if current_max > max_side or upscale:\n        scale = max_side / current_max\n        new_size = (max(1, round(width * scale)), max(1, round(height * scale)))\n        return image.resize(new_size, RESAMPLE)\n    return image.copy()\n\n\ndef preprocess_image(image_path: str | Path, variant: str = "raw") -> PreprocessResult:\n    image = load_rgb_image(image_path)\n    if variant == "raw":\n        processed = image\n    elif variant == "upscale":\n        processed = _resize_by_max_side(image, max_side=1024, upscale=True)\n    elif variant == "contrast":\n        processed = ImageEnhance.Contrast(image).enhance(1.6)\n    elif variant == "high_res":\n        processed = _resize_by_max_side(image, max_side=1568, upscale=False)\n    else:\n        raise ValueError(f"unknown preprocessing variant: {variant}")\n    return PreprocessResult(image=processed, name=variant, final_size=processed.size)\n\n\ndef select_preprocess_name(image_id: object) -> str:\n    image_id_text = str(image_id)\n    if image_id_text in LOW_RES_IMAGE_IDS:\n        return "upscale"\n    if image_id_text in CONTRAST_IMAGE_IDS:\n        return "contrast"\n    if image_id_text in HIGH_RES_IMAGE_IDS:\n        return "high_res"\n    return "raw"\n\n\ndef save_preprocessed_image(result: PreprocessResult, output_dir: str | Path, image_id: object) -> Path:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    output_path = output_dir / f"{image_id}_{result.name}.jpg"\n    result.image.convert("RGB").save(output_path, format="JPEG", quality=95)\n    return output_path\n', 'math_vqa/prompts.py': 'from __future__ import annotations\n\n\nBASE_PROMPT = """You are solving a Thai/English math problem from an image.\nRead all text, diagrams, graphs, shapes, and answer choices carefully.\nReturn only the final answer.\nDo not explain.\nDo not include "answer:".\nUse Arabic digits when possible."""\n\nDIAGRAM_PROMPT = """You are solving a math problem from an image.\nThe image may contain Thai text, English text, diagrams, graphs, shapes, or visual answer choices.\nUse the visual information, not only OCR text.\nReturn only the final answer."""\n\nSTRICT_FORMAT_PROMPT = """Return exactly one short answer string.\nNo explanation.\nNo units unless the unit is necessary to distinguish the answer.\nUse plain fractions such as 17/10.\nUse sqrt notation for radicals."""\n\nDIAGRAM_IMAGE_IDS = {"451", "569"}\nPROMPTS = {\n    "base": f"{BASE_PROMPT}\\n\\n{STRICT_FORMAT_PROMPT}",\n    "diagram": f"{DIAGRAM_PROMPT}\\n\\n{STRICT_FORMAT_PROMPT}",\n    "strict": STRICT_FORMAT_PROMPT,\n}\n\n\ndef select_prompt_name(image_id: object) -> str:\n    if str(image_id) in DIAGRAM_IMAGE_IDS:\n        return "diagram"\n    return "base"\n\n\ndef build_prompt(prompt_name: str = "base", few_shot_examples: list[dict[str, str]] | None = None) -> str:\n    if prompt_name not in PROMPTS:\n        raise ValueError(f"unknown prompt name: {prompt_name}")\n    prompt = PROMPTS[prompt_name]\n    if not few_shot_examples:\n        return prompt\n    example_lines = ["Examples of accepted final-answer strings:"]\n    for example in few_shot_examples:\n        example_lines.append(f"- id {example[\'id\']}: {example[\'answer\']}")\n    return f"{prompt}\\n\\n" + "\\n".join(example_lines)\n', 'math_vqa/ollama_client.py': 'from __future__ import annotations\n\nimport base64\nfrom pathlib import Path\nimport time\nfrom typing import Any\n\nimport requests\n\n\nDEFAULT_OLLAMA_HOST = "http://localhost:11434"\n\n\ndef encode_image_base64(image_path: str | Path) -> str:\n    return base64.b64encode(Path(image_path).read_bytes()).decode("ascii")\n\n\ndef assert_ollama_ready(host: str = DEFAULT_OLLAMA_HOST, timeout: int = 5) -> None:\n    try:\n        response = requests.get(f"{host}/api/tags", timeout=timeout)\n        response.raise_for_status()\n    except Exception as exc:\n        raise RuntimeError(f"Ollama is not ready at {host}: {exc}") from exc\n\n\ndef _extract_text_response(body: dict[str, Any]) -> str:\n    message = body.get("message")\n    if isinstance(message, dict) and message.get("content") is not None:\n        return str(message["content"])\n    if body.get("response") is not None:\n        return str(body["response"])\n    raise RuntimeError("Ollama response did not include message.content or response")\n\n\ndef query_llava(\n    image_path: str | Path,\n    prompt: str,\n    model: str = "llava",\n    host: str = DEFAULT_OLLAMA_HOST,\n    timeout: int = 600,\n    retries: int = 1,\n    retry_sleep: float = 5.0,\n) -> str:\n    payload = {\n        "model": model,\n        "stream": False,\n        "messages": [\n            {\n                "role": "user",\n                "content": prompt,\n                "images": [encode_image_base64(image_path)],\n            }\n        ],\n    }\n    attempts = max(0, retries) + 1\n    for attempt in range(attempts):\n        try:\n            response = requests.post(f"{host}/api/chat", json=payload, timeout=timeout)\n            response.raise_for_status()\n            body = response.json()\n            if not isinstance(body, dict):\n                raise RuntimeError("Ollama response was not a JSON object")\n            return _extract_text_response(body)\n        except requests.Timeout as exc:\n            if attempt < attempts - 1:\n                time.sleep(retry_sleep)\n                continue\n            raise RuntimeError(\n                f"Ollama request timed out after {timeout} seconds for {image_path}. "\n                "Increase OLLAMA_REQUEST_TIMEOUT, reduce image resolution/model size, or rely on fallback logging."\n            ) from exc\n        except requests.RequestException as exc:\n            if attempt < attempts - 1:\n                time.sleep(retry_sleep)\n                continue\n            raise RuntimeError(f"Ollama request failed for {image_path}: {exc}") from exc\n\n    raise RuntimeError("Ollama request failed without a captured exception")\n', 'math_vqa/submission.py': 'from __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport pandas as pd\n\n\n@dataclass(frozen=True)\nclass PredictionRecord:\n    id: str\n    image_path: str\n    raw_prediction: str\n    clean_answer: str\n    prompt_name: str\n    preprocess_name: str\n    final_size: str\n    runtime_seconds: float\n    inference_error: str\n    used_fallback: bool\n\n\ndef build_submission_frames(\n    records: list[PredictionRecord],\n    sample_submission: pd.DataFrame,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    record_by_id = {str(record.id): record for record in records}\n    sample_ids = [str(image_id) for image_id in sample_submission["id"].tolist()]\n    missing_ids = [image_id for image_id in sample_ids if image_id not in record_by_id]\n    if missing_ids:\n        raise ValueError(f"missing predictions for ids: {missing_ids[:10]}")\n\n    submission_rows = [\n        {"id": image_id, "answer": str(record_by_id[image_id].clean_answer)}\n        for image_id in sample_ids\n    ]\n    raw_rows = [asdict(record) for record in records]\n    submission_df = pd.DataFrame(submission_rows, columns=["id", "answer"])\n    raw_df = pd.DataFrame(\n        raw_rows,\n        columns=[\n            "id",\n            "image_path",\n            "raw_prediction",\n            "clean_answer",\n            "prompt_name",\n            "preprocess_name",\n            "final_size",\n            "runtime_seconds",\n            "inference_error",\n            "used_fallback",\n        ],\n    )\n    return submission_df, raw_df\n\n\ndef validate_submission(sample_submission: pd.DataFrame, submission: pd.DataFrame) -> None:\n    if list(submission.columns) != ["id", "answer"]:\n        raise ValueError(f"submission columns must be exactly [\'id\', \'answer\'], got {list(submission.columns)}")\n    if len(submission) != len(sample_submission):\n        raise ValueError(f"submission row count must be {len(sample_submission)}, got {len(submission)}")\n\n    sample_ids = [str(value) for value in sample_submission["id"].tolist()]\n    submission_ids = [str(value) for value in submission["id"].tolist()]\n    if submission_ids != sample_ids:\n        raise ValueError("submission row order must match sample_submission.csv")\n    if len(set(submission_ids)) != len(submission_ids):\n        raise ValueError("submission ids must be unique")\n    if submission["answer"].isna().any():\n        raise ValueError("submission has null answers")\n\n    empty_mask = submission["answer"].astype(str).str.strip().eq("")\n    if empty_mask.any():\n        empty_ids = submission.loc[empty_mask, "id"].astype(str).tolist()\n        raise ValueError(f"submission has empty answers for ids: {empty_ids[:10]}")\n\n\ndef write_outputs(\n    records: list[PredictionRecord],\n    sample_submission: pd.DataFrame,\n    output_dir: str | Path,\n) -> tuple[Path, Path]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    submission_df, raw_df = build_submission_frames(records, sample_submission)\n    validate_submission(sample_submission, submission_df)\n\n    submission_path = output_dir / "submission.csv"\n    raw_path = output_dir / "raw_predictions.csv"\n    submission_df.to_csv(submission_path, index=False)\n    raw_df.to_csv(raw_path, index=False)\n    return submission_path, raw_path\n'}
MISSING_MATERIALIZED_MODULES = []

for relative_path, source in MATERIALIZED_MODULES.items():
    target = RUNTIME_SRC / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

sys.path.insert(0, str(RUNTIME_SRC))
print(f"Materialized {len(MATERIALIZED_MODULES)} math_vqa module files into {RUNTIME_SRC}")
if MISSING_MATERIALIZED_MODULES:
    print("Missing planned module sources at notebook-build time:", MISSING_MATERIALIZED_MODULES)
    print("Rerun scripts/build_notebook.py after Tasks 1-5 complete to embed the full runtime package.")


In [ ]:
import pandas as pd

from math_vqa.data import CompetitionPaths, answer_prior, load_competition_frames, validate_data_files

paths = CompetitionPaths(DATA_ROOT)
train_df, test_df, sample_df = load_competition_frames(paths)
validate_data_files(paths, train_df, test_df, sample_df)
fallback_answer = answer_prior(train_df)

print(f"train rows: {len(train_df)}")
print(f"test rows: {len(test_df)}")
print(f"sample rows: {len(sample_df)}")
print(f"fallback answer: {fallback_answer}")


In [ ]:
answer_summary = train_df["answer"].astype(str).value_counts().head(10)
print("Top train answers:")
print(answer_summary)
print(f"Unique train answers: {train_df['answer'].nunique()}")


In [ ]:
from tqdm.auto import tqdm
import time

from math_vqa.data import resolve_image_path
from math_vqa.evaluation import normalized_accuracy
from math_vqa.ollama_client import assert_ollama_ready, query_llava
from math_vqa.postprocess import clean_model_answer
from math_vqa.preprocessing import preprocess_image, save_preprocessed_image, select_preprocess_name
from math_vqa.prompts import build_prompt, select_prompt_name
from math_vqa.submission import PredictionRecord, write_outputs

assert_ollama_ready(OLLAMA_HOST)
PREPARED_IMAGE_DIR = OUTPUT_DIR / "prepared_images"

def predict_image_record(image_id, image_path_value):
    started_at = time.time()
    image_path = resolve_image_path(paths, image_path_value)
    preprocess_name = select_preprocess_name(image_id)
    prompt_name = select_prompt_name(image_id)
    preprocess_result = preprocess_image(image_path, preprocess_name)
    prepared_path = save_preprocessed_image(preprocess_result, PREPARED_IMAGE_DIR, image_id)
    inference_error = ""
    try:
        raw_prediction = query_llava(
            prepared_path,
            build_prompt(prompt_name),
            model=MODEL_NAME,
            host=OLLAMA_HOST,
            timeout=OLLAMA_REQUEST_TIMEOUT,
            retries=OLLAMA_RETRIES,
        )
        cleaned = clean_model_answer(raw_prediction, fallback_answer)
    except Exception as exc:
        inference_error = f"{type(exc).__name__}: {exc}"
        raw_prediction = f"ERROR: {inference_error}"
        cleaned = clean_model_answer("", fallback_answer)
    runtime_seconds = round(time.time() - started_at, 3)
    return PredictionRecord(
        id=str(image_id),
        image_path=str(image_path_value),
        raw_prediction=raw_prediction,
        clean_answer=cleaned.answer,
        prompt_name=prompt_name,
        preprocess_name=preprocess_name,
        final_size=f"{preprocess_result.final_size[0]}x{preprocess_result.final_size[1]}",
        runtime_seconds=runtime_seconds,
        inference_error=inference_error,
        used_fallback=cleaned.used_fallback,
    )


In [ ]:
smoke_row = train_df.iloc[0]
smoke_record = predict_image_record(smoke_row["id"], smoke_row["image_path"])
print(
    {
        "id": smoke_row["id"],
        "truth": smoke_row["answer"],
        "raw_prediction": smoke_record.raw_prediction,
        "clean_answer": smoke_record.clean_answer,
        "preprocess_name": smoke_record.preprocess_name,
        "final_size": smoke_record.final_size,
        "runtime_seconds": smoke_record.runtime_seconds,
        "inference_error": smoke_record.inference_error,
        "used_fallback": smoke_record.used_fallback,
    }
)


In [ ]:
RUN_HOLDOUT_VALIDATION = False
HOLDOUT_ROWS = 20

if RUN_HOLDOUT_VALIDATION:
    holdout_df = train_df.sample(n=min(HOLDOUT_ROWS, len(train_df)), random_state=42)
    holdout_predictions = []
    holdout_truths = []
    for row in tqdm(holdout_df.itertuples(index=False), total=len(holdout_df), desc="holdout"):
        record = predict_image_record(row.id, row.image_path)
        holdout_predictions.append(record.clean_answer)
        holdout_truths.append(row.answer)
    print(f"Holdout normalized accuracy: {normalized_accuracy(holdout_predictions, holdout_truths):.4f}")
else:
    print("Holdout validation disabled for this run; train smoke test above was executed.")


In [ ]:
records = []
for row in tqdm(test_df.itertuples(index=False), total=len(test_df), desc="test inference"):
    records.append(predict_image_record(row.id, row.image_path))

submission_path, raw_path = write_outputs(records, sample_df, OUTPUT_DIR)
print(f"Wrote {submission_path}")
print(f"Wrote {raw_path}")
assert str(submission_path) == "/content/math-vqa-output/submission.csv" or submission_path.name == "submission.csv"
assert str(raw_path) == "/content/math-vqa-output/raw_predictions.csv" or raw_path.name == "raw_predictions.csv"


In [ ]:
experiment_log = pd.DataFrame(
    [
        {
            "run": "001",
            "model": MODEL_NAME,
            "setup": "Ollama HTTP API",
            "preprocessing": "image-id selector: raw/upscale/contrast/high_res",
            "prompt": "image-id selector: base/diagram + strict formatting",
            "postprocessing": "prefix cleanup, Thai digit conversion, empty-output fallback",
            "local_score": "",
            "public_lb": "",
            "notes": (
                f"fallback_count={sum(record.used_fallback for record in records)}; "
                f"error_count={sum(bool(record.inference_error) for record in records)}; "
                f"timeout_seconds={OLLAMA_REQUEST_TIMEOUT}; retries={OLLAMA_RETRIES}; output_dir={OUTPUT_DIR}"
            ),
        }
    ]
)
experiment_log_path = OUTPUT_DIR / "experiment_log.csv"
experiment_log.to_csv(experiment_log_path, index=False)
print(f"Wrote {experiment_log_path}")
